# Lecture 2 — Fit a Gamma model through the exponential family

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/AdvancedStatisticsCourse_36725561/blob/main/notebooks/lecture2_gamma.ipynb)


Fit a Gamma model to recorded summer storm rainfall. Then combine it with the lecture's Poisson count model by **marginalizing over the unknown number of storms**, and **summarize** the resulting annual-maximum distribution using return-level quantiles. Our illustrative design quantities are the **10-, 20- and 100-year return levels** of network-average storm rainfall.

| Section | What we accomplish | Contribution to the rainfall question |
|---|---|---|
| 1. Construct the distribution | Express Gamma through its statistics, natural parameters and log normalizer | Define the model whose parameters we will learn |
| 2. Summarize the observations | Calculate the empirical statistics used for fitting | Extract the rainfall data's contribution to estimation |
| 3. Fit the distribution | Match model expectations to empirical statistics | Learn the Gamma shape and rate |
| 4. Marginalize, then summarize | Average over possible storm counts; extract quantiles from the annual-maximum distribution | Obtain and interpret the modeled 10-, 20- and 100-year return levels |

There are **9 questions: six code and three written**. Save your own copy and work from top to bottom. Execute **Code to run** cells; fill the answer cell under each question. AI assistance is allowed: include the nearby equations and check the answer before using it. Once finished, restart and run all cells. A standard CPU runtime is sufficient.

## 1. Build the distribution from its exponential-family form

### *Explanation to read*

#### Define the distribution we need to fit

The 227 measurements are positive network-average rainfall amounts, in inches, for recorded summer storms in southern Illinois, 1960–1964. The original data consist of five separate annual files containing 48, 48, 56, 37 and 38 observations respectively. The combined public CSV retains those 227 values in the exact original year order. It has no year column, but we can recover the grouping from the verified original file boundaries.

We treat storm amounts as conditionally independent draws from a common Gamma distribution. This is an assumption, not a verified property of the rainfall or its recording. The Poisson model from the lecture describes how many recorded events occur in a summer; the Gamma model describes the positive rainfall amount of one recorded event.

Gamma has positive support and a flexible right-skewed shape. Its shape and rate will be inferred from the observed storm amounts.

### *Code to run*

#### Data and numerical tools

The existing public CSV contains the 227 positive rainfall measurements in their original chronological-file order (1960 through 1964), but does not have a year column. The original annual files and the verified two-column year-labeled version are documented in the course data provenance. urlopen accesses the public CSV; np.loadtxt skips its header and loads the values into rain. NumPy handles arrays; SciPy supplies numerical tools; plotting is supplied.

In [ ]:
import numpy as np
from urllib.request import urlopen
import matplotlib.pyplot as plt
from scipy.special import gammaln, digamma
from scipy.optimize import root
from scipy.integrate import quad

data_url = (
    "https://raw.githubusercontent.com/opherdonchin/"
    "AdvancedStatisticsCourse_36725561/main/data/illinois_rainfall_inches.csv"
)
with urlopen(data_url, timeout=30) as data_file:
    rain = np.loadtxt(data_file, delimiter=",", skiprows=1)
n = len(rain)
assert n == 227 and np.all(np.isfinite(rain)) and np.all(rain > 0)
print(f"{n} positive amounts; range {rain.min():.3f}–{rain.max():.2f} inches")
from scipy.stats import gamma


### *Questions to answer*

#### 1. Rewrite the Gamma density as an exponential — written answer

Use $h(x)=1$ on $x>0$. Rewrite the supplied Gamma density as
$$
p(x\mid\alpha,\beta)
=h(x)\exp\!\left[f_1(x,\alpha,\beta)+f_2(\alpha,\beta)\right].
$$
Keep $\alpha-1$ together as one coefficient. Put all terms containing $x$ in $f_1$ and all parameter-only terms in $f_2$. Specify the two functions.

- answer here

### *Explanation to read*

#### Connect your decomposition to the lecture’s representation

Recall the exponential-family form
$$p(x\mid\eta)=h(x)\exp\!\left(\eta^\mathsf{T}T(x)-A(\eta)\right).$$
Using $h(x)=1$ as in Question 1, match your two terms to
$$
\eta^\mathsf{T}T(x)=f_1(x,\alpha,\beta),\qquad
-A(\eta)=f_2(\alpha,\beta).
$$

### *Questions to answer*

#### 2. Identify the natural parameters and statistics — written answer

Find vectors $\eta$ and $T(x)$ such that
$$\eta^\mathsf{T}T(x)=f_1(x,\alpha,\beta),$$
using your answer to Question 1.

- answer here

### *Questions to answer*

#### 3. Implement the natural parameters — code

Write `natural_parameters(shape, rate)` returning your $\eta$ as a NumPy array, with `shape` $=\alpha$ and `rate` $=\beta$. Use the order $T(x)=(\log x,x)^\mathsf{T}$ for all code below.

In [ ]:
# answer here

### *Explanation to read*

#### Identify the log normalizer in your decomposition

The parameter-only term $f_2(\alpha,\beta)$ from Question 1 corresponds to $-A(\eta)$.

Write the natural-parameter vector as
$$\eta=\begin{pmatrix}\eta_0\\\eta_1\end{pmatrix}
=\begin{pmatrix}\alpha-1\\-\beta\end{pmatrix}.$$
Thus $\alpha=\eta_0+1$ and $\beta=-\eta_1$, with $\eta_0>-1$ and $\eta_1<0$. In Python, these components are `eta[0]` and `eta[1]`.

You can use `gammaln(a)` to compute $\log\Gamma(a)$. It was imported from `scipy.special` above. Use `np.log` for other logarithms.

### *Questions to answer*

#### 4. Implement the log normalizer — code

Use $f_2(\alpha,\beta)$ from your answer to Question 1 to write `log_partition(eta)` returning $A(\eta)$. You may recover shape/rate inside the function.

In [ ]:
# answer here

### *Code to run*

#### Define the log density

This supplied function combines your natural parameters, observation statistics and log normalizer to evaluate $\log p(x\mid\eta)$ for positive $x$.

In [ ]:
def gamma_logpdf(x, eta):
    return eta[0] * np.log(x) + eta[1] * x - log_partition(eta)

### *Explanation to read*

#### Check that the density integrates to one

A probability density must satisfy
$$\int_0^\infty p(x\mid\eta)\,dx=1.$$
The term $A(\eta)$ provides this normalization. We will check the integral numerically at shape 2, rate 3, using the log density just defined. These are candidate parameters, not a fit to the rainfall data.

### *Code to run*

#### Integrate the candidate density

`np.exp` converts the log density to a density. `quad` returns its numerical integral and an estimated integration error.

In [ ]:
eta_candidate = natural_parameters(2.0, 3.0)
mass, error = quad(lambda x: np.exp(gamma_logpdf(x, eta_candidate)), 0, np.inf)
assert abs(mass - 1) < 1e-7
print(f"Candidate density integral: {mass:.8f}")

### *Explanation to read*

#### Section summary

The first step of our route is complete: you have expressed the Gamma model through its statistics, natural parameters and log normalizer. Its shape and rate are still unknown.

## 2. Reduce the data to the quantities fitting needs

### *Explanation to read*

#### From individual observations to the sample likelihood

Let $x_1,\ldots,x_n$ be the observed rainfall amounts. Assuming independent observations from the same distribution, their joint density is
$$
p(x_1,\ldots,x_n\mid\eta)=\prod_{i=1}^n p(x_i\mid\eta).
$$
Substitute the exponential-family expression from Section 1:
$$
\begin{aligned}
p(x_1,\ldots,x_n\mid\eta)
&=\prod_{i=1}^n\left[h(x_i)\exp\!\left(\eta^\mathsf{T}T(x_i)-A(\eta)\right)\right]\\
&=\left[\prod_{i=1}^n h(x_i)\right]
\exp\!\left(\sum_{i=1}^n\left[\eta^\mathsf{T}T(x_i)-A(\eta)\right]\right)\\
&=\left[\prod_{i=1}^n h(x_i)\right]
\exp\!\left(\eta^\mathsf{T}\sum_{i=1}^n T(x_i)-nA(\eta)\right).
\end{aligned}
$$
The same $\eta$ applies to every observation, and $A(\eta)$ appears $n$ times. For our Gamma representation, every $h(x_i)=1$, so the product of these terms is one.

The observations therefore enter this likelihood through $\sum_i T(x_i)$. We will calculate their average,
$$\overline T=\frac1n\sum_{i=1}^n T(x_i),$$
so that the accumulated statistic is $n\overline T$. Keep the logarithmic-first order from Section 1.

### *Questions to answer*

#### 5. Calculate the empirical statistic average — code

Compute the two-component empirical average of your $T(x)$ from `rain`, storing it as the NumPy vector `t_bar`.

In [ ]:
# answer here

### *Code to run*

#### Display the empirical target

The entries are the mean log amount and mean amount; these are the targets for the model expectations in Section 3.

In [ ]:
print("Empirical [mean(log X), mean(X)]:", t_bar)

### *Explanation to read*

#### Use the empirical averages in the log likelihood

Taking the logarithm of the sample density above and substituting $\sum_i T(x_i)=n\overline T$ gives
$$
\begin{aligned}
\ell(\eta)
&=\log p(x_1,\ldots,x_n\mid\eta)\\
&=\eta^\mathsf{T}\sum_{i=1}^n T(x_i)-nA(\eta)\\
&=n\left[\eta^\mathsf{T}\overline T-A(\eta)\right].
\end{aligned}
$$
Thus $n$ and $\overline T$ determine the entire Gamma likelihood function.

Differentiating with respect to $\eta$ gives
$$\nabla_\eta\ell(\eta)=n\left[\overline T-\nabla_\eta A(\eta)\right].$$
Setting $\nabla_\eta\ell(\eta)=0$, we get
$$\nabla_\eta A(\eta)=\overline T.$$

### *Code to run*

#### Define the log likelihood

This function evaluates the sample log likelihood from `n`, `t_bar` and your log normalizer.

In [ ]:
def log_likelihood(eta, n, t_bar):
    return n * (eta @ t_bar - log_partition(eta))

### *Explanation to read*

#### Compare the two ways to evaluate the likelihood

We can evaluate the same sample log likelihood by adding the individual log densities or by using $n$ and $\overline T$:
$$\sum_{i=1}^n\log p(x_i\mid\eta)
=n\left[\eta^\mathsf{T}\overline T-A(\eta)\right].$$
The two calculations should agree at the same parameter vector.

### *Code to run*

#### Check agreement at the candidate parameters

The code compares the raw-data sum with your statistic-based calculation at `eta_candidate`.

In [ ]:
ll_raw = gamma_logpdf(rain, eta_candidate).sum()
ll_reduced = log_likelihood(eta_candidate, n, t_bar)
assert np.isclose(ll_raw, ll_reduced)
print(f"Raw sum: {ll_raw:.6f}; statistic-based value: {ll_reduced:.6f}")

### *Explanation to read*

#### Section summary

The second step is complete: you have computed the two empirical statistics. Together with the sample size, they retain all the observed information needed to evaluate this Gamma likelihood.

## 3. Fit by matching model and empirical statistics

### *Explanation to read*

#### Connect the likelihood condition to model expectations

Use the same natural coordinates as in Section 1:
$$
\eta=\begin{pmatrix}\eta_0\\\eta_1\end{pmatrix}
=\begin{pmatrix}\alpha-1\\-\beta\end{pmatrix},
\qquad \alpha=\eta_0+1,\quad\beta=-\eta_1.
$$
Recall the log normalizer:
$$A(\eta)=\log\Gamma(\eta_0+1)-(\eta_0+1)\log(-\eta_1),
\qquad \eta_0>-1,\quad\eta_1<0.$$

Section 2 gave the zero-gradient equation $\nabla_\eta A(\eta)=\overline T$. The lecture's identity $\nabla_\eta A(\eta)=E_\eta[T(X)]$ tells us what its left-hand side represents. Substituting gives
$$E_\eta[T(X)]=\overline T.$$
Fitting therefore matches the model's expected statistics to their observed averages.

### *Explanation to read*

#### The Gamma model's expected statistics

Define $\psi(a)=\frac{d}{da}\log\Gamma(a)$; Python's `digamma(a)` evaluates this function. Differentiating the log normalizer gives
$$
\nabla_\eta A(\eta)=
\begin{pmatrix}
\psi(\eta_0+1)-\log(-\eta_1)\\
-(\eta_0+1)/\eta_1
\end{pmatrix}.
$$
Using $\alpha=\eta_0+1$ and $\beta=-\eta_1$, the same natural-coordinate gradient can be expressed using shape and rate:
$$
E_\eta[T(X)]=
\begin{pmatrix}E_\eta[\log X]\\E_\eta[X]\end{pmatrix}
=
\begin{pmatrix}\psi(\alpha)-\log\beta\\\alpha/\beta\end{pmatrix}.
$$
The shape/rate expression is convenient for the code below.

### *Questions to answer*

#### 6. Implement the model expectations — code

Using the supplied expected-statistic formula, write `expected_statistics(eta)` returning the two-component NumPy vector $E_\eta[T(X)]$. Recover shape and rate as `eta[0] + 1` and `-eta[1]`; use `digamma` where needed.

In [ ]:
# answer here

### *Code to run*

#### Compare candidate expectations with the data

The data vector stays fixed. Fitting will change the parameters so that model expectations match it.

In [ ]:
print("Data target:         ", t_bar)
print("Candidate expectation:", expected_statistics(eta_candidate))

### *Explanation to read*

#### Specify the equations to solve

The fitting condition is
$$E_{\widehat\eta}[T(X)]-\overline T=0.$$
The two equations determine shape and rate together. Matching only the ordinary mean would leave every pair with $\alpha/\beta=\overline x$ possible. The mean log amount supplies the second constraint.

### *Code to run*

#### Define the discrepancy for the solver

The solver needs a function whose zero expresses the fitting condition. This supplied function subtracts the observed averages from your model expectations.

In [ ]:
def moment_residual(eta):
    return expected_statistics(eta) - t_bar

### *Code to run*

#### Keep solver parameters in the valid domain

The solver will search over $u=(\log\alpha,\log\beta)$. This supplied wrapper converts its coordinates to natural parameters before evaluating `moment_residual`.

In [ ]:
def residual_in_solver_coordinates(u):
    shape, rate = np.exp(u)
    eta = natural_parameters(shape, rate)
    return moment_residual(eta)

initial_guess = np.log([1.0, 1.0])
print("Initial residual:", residual_in_solver_coordinates(initial_guess))

### *Explanation to read*

#### Call the numerical solver

`root(function, initial_guess, tol=1e-10)` searches for a zero vector. It returns a result whose `.x` contains the solution coordinates and whose `.success` and `.message` describe termination. The supplied wrapper is the function the solver should use; the iteration algorithm is already implemented by SciPy.

### *Questions to answer*

#### 7. Solve the fitting equations — code

Call `root` with `residual_in_solver_coordinates`, the supplied `initial_guess` and `tol=1e-10`. Store the result as `fit`.

In [ ]:
# answer here

### *Code to run*

#### Inspect the fitted model

This converts the solution back to shape/rate, checks the residual and compares model expectations with the data targets.

In [ ]:
if not fit.success:
    raise RuntimeError(fit.message)
shape_hat, rate_hat = np.exp(fit.x)
eta_hat = natural_parameters(shape_hat, rate_hat)
assert np.max(np.abs(moment_residual(eta_hat))) < 1e-8
print(f"Fitted shape: {shape_hat:.6f}; rate: {rate_hat:.6f} per inch")
print("Data target:       ", t_bar)
print("Fitted expectation:", expected_statistics(eta_hat))
print(f"Candidate log likelihood: {log_likelihood(eta_candidate,n,t_bar):.6f}")
print(f"Fitted log likelihood:    {log_likelihood(eta_hat,n,t_bar):.6f}")

### *Explanation to read*

#### Section summary

The third step is complete: you implemented the model expectations and called the solver to find the fitted Gamma parameters. The fitted model expectations now agree with the empirical statistics.

## 4. Marginalize storm counts, then summarize annual rainfall risk

### *Explanation to read*

#### Two familiar operations on a new model

The lecture modeled the annual number of recorded storms with a Poisson distribution. The five recovered counts were 48, 48, 56, 37 and 38; their maximum-likelihood Poisson rate is $\widehat m=45.4$ storms per summer. We have now fitted a Gamma distribution to the 227 positive storm amounts.

We want the probability that **at least one storm in a summer exceeds rainfall threshold $r$**. The number of storms is not known for a future summer, so the calculation requires two operations:

**1. Marginalize over the number of storms.** Write $N$ for the annual storm count, $F_\Gamma(r)$ for the fitted Gamma CDF of a single storm amount, and $M$ for the maximum amount in the summer (define $M=0$ when $N=0$). Assume the annual count is Poisson with fixed parameter $m>0$, storm amounts are independent draws from the fitted Gamma distribution given $N$, and the count and amounts are independent. For $r\ge0$, if we knew $N=n$, then all $n$ storms would have to be at or below $r$ for the maximum to be at or below $r$:

$$P(M\le r\mid N=n,m)=F_\Gamma(r)^n.$$

For an unknown $N$, use the **law of total probability**, weighting this conditional probability by each possible annual count:

$$
\begin{aligned}
P(M\le r\mid m)
&=\sum_{n=0}^{\infty}P(N=n\mid m)\,F_\Gamma(r)^n\\
&=e^{-m}\sum_{n=0}^{\infty}\frac{[mF_\Gamma(r)]^n}{n!}\\
&=e^{-m}e^{mF_\Gamma(r)}\\
&=\exp\{-m[1-F_\Gamma(r)]\}.
\end{aligned}
$$

The third line uses the exponential power series $\sum_{n=0}^{\infty}z^n/n!=e^z$. This is an **exact marginalization over Poisson storm counts**. It is *not* a substitution of the mean count into the probability for a fixed number of storms: in general,

$$\mathbb E[F_\Gamma(r)^N]\ne F_\Gamma(r)^{\mathbb E[N]}.$$

We then **plug in** the fitted rate $\widehat m=45.4$ and fitted Gamma parameters. This is a separate approximation: we treat estimated parameters as known instead of integrating over our uncertainty about them. The modeled probability of at least one exceedance is therefore

$$P(M>r)\approx 1-\exp[-\widehat m(1-F_{\widehat\Gamma}(r))].$$

The approximation symbol refers to the *use of fitted parameters*, not to the Poisson marginalization. There is no attempt to infer a posterior distribution for the parameters in this notebook.

**2. Summarize the annual-maximum distribution.** Now we have the fitted CDF of the summer maximum. A $T$-year return level $r_T$ is its upper quantile with annual exceedance probability $1/T$:

$$P(M>r_T)\approx\frac1T,\qquad F_M(r_T)\approx1-\frac1T.$$

Solving the annual-maximum CDF for the corresponding *single-storm* Gamma quantile gives

$$F_{\widehat\Gamma}(r_T)=1+\frac{\log(1-1/T)}{\widehat m}.$$

We will compute return levels for $T=10,20,100$ summers. These are descriptive summaries of the fitted **summer-maximum** distribution, not summaries of individual-storm rainfall and not validated engineering protection levels.

### *Code to run*

#### Recover annual counts and fit Poisson

The five original files contain 48, 48, 56, 37 and 38 recorded events, one file per summer in 1960–1964. Because the existing public CSV has only rainfall amounts, we recover each observation's year using these verified file-block lengths. The two-column year-labeled source in the development repository confirms this grouping. Check that the total count matches the loaded amounts, then fit the Poisson annual rate by averaging the five counts. The sample variance is a descriptive diagnostic, not a formal model validation.

In [ ]:
annual_years = np.arange(1960, 1965)
annual_counts = np.array([48, 48, 56, 37, 38])
storm_year = np.repeat(annual_years, annual_counts)
assert len(rain) == len(storm_year) == annual_counts.sum() == 227
m_hat = annual_counts.mean()
count_variance = annual_counts.var(ddof=1)
for year, count in zip(annual_years, annual_counts):
    print(f"{year}: {count} recorded storms")
print(f"Poisson MLE: {m_hat:.1f} storms/summer; sample variance: {count_variance:.1f}")

### *Explanation to read*

#### Calculate the summary from the marginal distribution

We have already marginalized over the unknown annual number of storms to obtain the summer-maximum CDF. Our **summary** is an upper quantile of that distribution, defined by its return period.

The previous derivation converts the summer-maximum quantile into a quantile of the fitted *single-storm Gamma distribution*. `gamma.ppf(q, a=shape, scale=1/rate)` returns the storm amount corresponding to Gamma CDF probability `q`; in SciPy, `scale` is the reciprocal of the rate. This is how we calculate the annual-maximum return level, **not** an assertion that the annual maximum is Gamma-distributed.

### *Questions to answer*

#### 8. Summarize the annual maximum using return levels — code

After marginalizing over the Poisson storm count, summarize the resulting annual-maximum distribution using its 10-, 20- and 100-year return levels. Set `return_periods` to `np.array([10.0, 20.0, 100.0])`; use the fitted annual Poisson rate `m_hat`, the quantile formula above, and `gamma.ppf` with your fitted Gamma shape and rate to calculate `return_levels`.

In [ ]:
# answer here

### *Code to run*

#### Display the capacity thresholds

Each row gives the summer exceedance probability and associated rainfall threshold. All amounts are network-average storm rainfall in inches.

In [ ]:
for period, level in zip(return_periods, return_levels):
    print(f"{period:3.0f}-year level: {level:.3f} inches; summer exceedance {1/period:.1%}")
print(f"Largest observed storm average: {rain.max():.3f} inches; record: five summers")

### *Code to run*

#### Examine the fit and its extrapolation

The left panel shows the observed storm averages and fitted density (clipped at density 6 to keep the histogram visible near zero). The right shows the fitted summer-maximum exceedance curve and your return levels; it extrapolates beyond what five summers can establish.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x_grid = np.linspace(0.003, 2.5, 600)
axes[0].hist(rain, bins=np.linspace(0, 2.5, 26), density=True, alpha=0.5, label="Observed storms")
axes[0].plot(x_grid, np.exp(gamma_logpdf(x_grid, eta_hat)), label="Fitted Gamma")
axes[0].set(xlabel="Storm rainfall (inches)", ylabel="Density", ylim=(0, 6))
axes[0].legend()
r_grid = np.linspace(0.01, max(return_levels) * 1.15, 600)
summer_exceedance = -np.expm1(-m_hat * gamma.sf(r_grid, a=shape_hat, scale=1/rate_hat))
axes[1].semilogy(r_grid, summer_exceedance, label="Fitted summer exceedance")
axes[1].scatter(return_levels, 1/return_periods, label="Return levels")
axes[1].axvline(rain.max(), color="gray", linestyle=":", label="Largest observed storm")
axes[1].set(xlabel="Threshold (inches)", ylabel="Summer exceedance probability")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

### *Questions to answer*

#### 9. Assess the 100-year extrapolation — written answer

Why do the Gamma fit to positive storm amounts and the Poisson fit to five annual storm counts not establish that the modeled 100-year summer return level is reliable?

- answer here

### *Explanation to read*

#### What we learned to do

You fitted a Gamma distribution to positive storm amounts using exponential-family sufficient statistics and moment matching. You then returned to two general probabilistic operations from the course:

- **Marginalization:** starting from the conditional probability of no large storm given exactly $n$ storms, sum over the Poisson probabilities of every possible $n$ to obtain the annual-maximum distribution. The exponential simplification is exact for a Poisson count model.
- **Summarization:** use an upper quantile of that annual-maximum distribution to report the modeled 10-, 20- and 100-year return levels.

The fitted Poisson rate and Gamma parameters are **plug-in estimates**. Their uncertainty, recording completeness, independence, stationarity and far-tail adequacy were not established by fitting. This distinction between performing the probabilistic operations correctly and choosing an adequate model is part of interpreting the answer.

**Sources.** Murphy, *Probabilistic Machine Learning: An Introduction*, §2.7.5 and §3.4; *Advanced Topics*, §§2.4.3–2.4.5. The real-data example comes from [MIT 18.443 Project 2](https://live.ocw.mit.edu/ans7870/18/18.443/s15/projects/Rproject2_script2_gamma_MOMwithMLE.html); numeric observations are the five Rice-data mirrors recorded in the course repository. [Fonnesbeck’s rainfall notebook](https://github.com/fonnesbeck/pytenn2014_tutorial/blob/master/Part%202.%20Statistical%20Data%20Modeling.ipynb) supplies an earlier computational precedent. This notebook uses shape/rate throughout and fits the averages of $X$ and $\log X$, rather than an ordinary mean/variance method-of-moments estimator.